# Language Subtitles — Gather Sources

First step of Language Subtitles: collects **two** raw text sources per target language —

1. **YouTube captions** (manual or auto-generated) — `{NOME}_yt_{lang}.srt`
2. **Auto-dubbed audio + Whisper transcription** — `{NOME}_audio_{lang}.wav` and `{NOME}_whisper_{lang}.srt`

Neither of these is synced to the master timing yet — that happens in `caption-multilang-generate.ipynb`,
which redistributes whichever source you choose (per language) onto the master caption's exact blocks/timing.

**Manual correction workflow** (same pattern as Single Subtitle):
1. Run this notebook — saves the raw files above to Drive.
2. Download whichever ones you want to review (last cell), correct locally.
3. Upload back to Drive, replacing the same file.
4. Run `caption-multilang-generate.ipynb` whenever you're ready — it always reads the current
   Drive files, never a stale local copy.

⚠️ **About YouTube downloads**: the YouTube video needs auto-dubbing / caption tracks available for
your target languages — not all videos have all languages. Missing languages are skipped with a
warning, not a hard failure. Downloads from cloud IPs (Colab included) are sometimes blocked by
YouTube ("Sign in to confirm you're not a bot") — see the Setup cell for the cookies workaround.


In [1]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📦 SETUP — packages, Google Drive, and modules (run once per session) ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── System packages ──────────────────────────────────────────────────────────
!apt-get -qq -y install ffmpeg curl > /dev/null 2>&1
print('✅ ffmpeg')

# ── Python packages ───────────────────────────────────────────────────────────
!pip install -q -U yt-dlp openai-whisper
print('✅ yt-dlp, openai-whisper')

# ── Mount Drive (unmount first to avoid a stuck session) ────────────────────
from google.colab import drive
try:
    drive.flush_and_unmount()
except Exception:
    pass
drive.mount('/content/drive', force_remount=True)
print('✅ Drive mounted')

# ── Copy modules from Drive to /content/pipeline ────────────────────────────
import shutil, os, sys, logging
from pathlib import Path

PASTA_DRIVE_RAIZ = "narrated_video"  # fixed for the whole project (same value as Configuration)
PASTA_MODULOS = Path(f"/content/drive/MyDrive/{PASTA_DRIVE_RAIZ}/pipeline/modulos")
DESTINO = Path("/content/pipeline")

if PASTA_MODULOS.exists():
    if DESTINO.exists():
        shutil.rmtree(DESTINO)
    shutil.copytree(PASTA_MODULOS, DESTINO)
    print(f"✅ {len(list(DESTINO.glob('*.py')))} modules copied from {PASTA_MODULOS}")

    # ── A cópia trouxe TODOS os módulos? ───────────────────────────────────────
    # "N módulos copiados" sozinho não quer dizer nada. E o modo de falhar aqui é
    # traiçoeiro: o Drive montado do Colab popula a listagem da pasta com atraso,
    # então um copytree logo depois do mount às vezes enxerga só parte dos
    # arquivos. Já aconteceu de copiar 13 de 31 -- com visto verde -- e o notebook
    # quebrar muito depois, num import, longe da causa.
    #
    # A conferência é de três pontas, porque a causa muda o conserto:
    #   manifesto  o que o repositório tem  (gravado pelo repositorio-sincronizar)
    #   Drive      o que chegou lá
    #   VM         o que a cópia desta célula trouxe
    _manifesto = PASTA_MODULOS / "_manifesto.txt"
    if not _manifesto.exists():
        print("   ⚠️  sem _manifesto.txt — rode o repositorio-sincronizar pra criá-lo")
    else:
        _esperados = {l.strip() for l in _manifesto.read_text().splitlines()
                      if l.strip() and not l.startswith("#")}
        _no_drive = {f.name for f in PASTA_MODULOS.glob("*.py")}
        _na_vm    = {f.name for f in DESTINO.glob("*.py")}

        _fora_do_drive = sorted(_esperados - _no_drive)
        _nao_copiados  = sorted((_esperados & _no_drive) - _na_vm)

        if _nao_copiados:
            # Estão no Drive mas não vieram: é a listagem preguiçosa do mount.
            # Uma segunda passada, com o mount já quente, costuma resolver.
            print(f"   ⏳ {len(_nao_copiados)} módulo(s) não vieram na 1ª passada — copiando de novo")
            for _n in _nao_copiados:
                shutil.copyfile(PASTA_MODULOS / _n, DESTINO / _n)
            _na_vm = {f.name for f in DESTINO.glob("*.py")}
            _nao_copiados = sorted((_esperados & _no_drive) - _na_vm)

        if _fora_do_drive:
            print(f"\n🚨 {len(_fora_do_drive)} módulo(s) não estão no DRIVE:")
            for _n in _fora_do_drive:
                print(f"     {_n}")
            raise SystemExit("Rode o repositorio-sincronizar.ipynb — o Drive está atrás do repositório.")
        if _nao_copiados:
            print(f"\n🚨 {len(_nao_copiados)} módulo(s) estão no Drive mas não copiaram:")
            for _n in _nao_copiados:
                print(f"     {_n}")
            raise SystemExit("Rode ESTA célula de novo — o Drive montado ainda estava acordando.")
        print(f"   ✅ os {len(_esperados)} módulos do manifesto estão na VM")
else:
    print(f"❌ Modules folder not found: {PASTA_MODULOS}")

if str(DESTINO) not in sys.path:
    sys.path.insert(0, str(DESTINO))

# ── Clear stale local data files from any previous run in this session ─────
# Modules above are always freshly copied (rmtree + copytree), but DATA files
# (.srt, .wav, .ass) downloaded/generated by earlier cells in this same
# session could still be sitting in /content — if you re-run after
# correcting something on Drive, you want THIS run to fetch everything
# fresh, not silently reuse an old local copy. This removes any leftover
# language-subtitle data files before starting.
padroes_para_limpar = ["*.srt", "*.wav", "*.ass"]
removidos = 0
for padrao in padroes_para_limpar:
    for arquivo in Path('/content').glob(padrao):
        arquivo.unlink()
        removidos += 1
print(f"✅ {removidos} stale local file(s) cleared — this run will fetch everything fresh from Drive")

logging.basicConfig(level=logging.INFO, format='%(asctime)s  %(name)-24s  %(levelname)s  %(message)s', datefmt='%H:%M:%S')
os.chdir('/content')
print('✅ Setup complete!')


✅ ffmpeg
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.8/183.8 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 29.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 109.4 MB/s eta 0:00:00
✅ yt-dlp, openai-whisper
Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive
✅ Drive mounted
✅ 13 modules copied from /content/drive/MyDrive/narrated_video/pipeline/modulos
✅ 0 stale local file(s) cleared — this run will fetch everything fresh from Drive
✅ Setup complete!


In [2]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  ⚙️  CONFIGURATION                                               ║
# ║  ✏️  Edit only this cell — same NOME_ORACAO as earlier steps     ║
# ╚══════════════════════════════════════════════════════════════════╝

# ── 1. VIDEO IDENTITY (must match video-base-*.ipynb / caption-single-generate.ipynb) ──
NOME_ORACAO = "40_Matt_02"

# ── 1b. MASTER LANGUAGE (must match caption-single-generate.ipynb for this video) ───
# PipelineConfig defaults IDIOMA_MESTRE to "pt" — override it here whenever
# this specific video's narration/master audio is NOT Portuguese. This
# matters a lot: without it, the master-language protection (which skips
# downloading/transcribing whatever IDIOMA_MESTRE is set to, since that
# language is already handled by caption-single-generate.ipynb) silently protects
# the WRONG language — it would skip "pt" by mistake and process "en" by
# mistake, exactly backwards from what you want.
IDIOMA_MESTRE = "en"

# ── 2. SOURCE YOUTUBE VIDEO ─────────────────────────────────────────────────
# The video that has the auto-dubbed audio tracks / captions you want to pull from.
URL_YOUTUBE = "https://www.youtube.com/watch?v=4vTN7tBG3a8"

# ── 3. TARGET LANGUAGES ──────────────────────────────────────────────────────
# Canonical codes used across the whole project (position/color/font/Whisper).
IDIOMAS_ALVO = ["en", "pt", "es", "fr", "ko"]

# ── 4. WHISPER MODEL (for transcribing the dubbed audio) ───────────────────
# Medido no 40_Matt_02 contra o roteiro: "base" deu 24 divergências / 0.9521 de
# similaridade, "small" deu 16 / 0.9692 -- e a diferença estava justamente nos
# nomes próprios (Herodes, Belém, Arquelau), que é do que a Bíblia é feita. As
# faixas aqui são dubladas e não estão em inglês, onde o "base" erra ainda mais.
# Com GPU ligada (Ambiente de execução → Alterar o tipo → GPU) custa ~1 min por
# idioma. Continua variável: troque se algum idioma específico pedir outro.
MODELO_WHISPER = "small"

# ── 5. YOUTUBE CAPTION CODE OVERRIDE (optional) ─────────────────────────────
# Only needed if a language's YouTube caption code differs from the
# canonical one above (the most common case is Chinese: internally "zh",
# but YouTube lists it as "zh-Hans" or "zh-Hant" — already set below).
# Run the diagnostic snippet (yt-dlp --list-subs URL) if unsure what a
# specific video uses.
CODIGO_LEGENDA_YOUTUBE = {
    "zh": "zh-Hans",   # use "zh-Hant" instead for Traditional Chinese
}

# ── 6. MANUAL AUDIO FORMAT OVERRIDE (optional) ──────────────────────────────
# The automatic filter (best audio matching the language) sometimes fails
# with "Requested format is not available" even when the track exists —
# YouTube's auto-dub tracks can be temporarily unstable. If that happens:
#   1. Run "yt-dlp -F <URL>" (or the diagnostic snippet) to see current IDs.
#   2. Find the row for that language (e.g. "251-11  ...  [pt-BR] ...").
#   3. Put the exact ID here for that language.
# Leave a language out to keep using the automatic filter.
FORMATO_MANUAL_AUDIO = {
    # "pt": "251-11",
}

# ── 7. DRIVE ROOT FOLDER ────────────────────────────────────────────────────
PASTA_DRIVE_RAIZ = "narrated_video"     # ⚠️ DO NOT CHANGE — fixed for the whole project

# ── CHECK ──────────────────────────────────────────────────────────────────
print("=" * 60)
print("⚙️  CONFIGURATION")
print("=" * 60)
print(f"   Video:           {NOME_ORACAO}")
print(f"   Master language: {IDIOMA_MESTRE}")
print(f"   YouTube URL:     {URL_YOUTUBE}")
print(f"   Target langs:    {IDIOMAS_ALVO}")
print(f"   Whisper model:   {MODELO_WHISPER}")
print(f"   YT caption code: {CODIGO_LEGENDA_YOUTUBE}")
print(f"   Manual audio fmt:{FORMATO_MANUAL_AUDIO or ' (none — using automatic filter)'}")
print(f"   Drive root:      {PASTA_DRIVE_RAIZ}")
print("=" * 60)
print("✅ Configuration ready — proceed to Initialization")


⚙️  CONFIGURATION
   Video:           40_Matt_02
   Master language: en
   YouTube URL:     https://www.youtube.com/watch?v=4vTN7tBG3a8
   Target langs:    ['en', 'pt', 'es', 'fr', 'ko']
   Whisper model:   base
   YT caption code: {'zh': 'zh-Hans'}
   Manual audio fmt: (none — using automatic filter)
   Drive root:      narrated_video
✅ Configuration ready — proceed to Initialization


In [3]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🚀 INITIALIZE PIPELINE                                          ║
# ╚══════════════════════════════════════════════════════════════════╝

import sys
from pathlib import Path

if '/content/pipeline' not in sys.path:
    sys.path.insert(0, '/content/pipeline')

from config import PipelineConfig
from language_captions_pipeline import LanguageCaptionsPipeline

config = PipelineConfig(
    NOME_ORACAO             = NOME_ORACAO,
    PASTA_DRIVE_RAIZ         = PASTA_DRIVE_RAIZ,
    IDIOMA_MESTRE            = IDIOMA_MESTRE,
    CODIGO_LEGENDA_YOUTUBE   = CODIGO_LEGENDA_YOUTUBE,
    FORMATO_MANUAL_AUDIO     = FORMATO_MANUAL_AUDIO,
)

pipeline = LanguageCaptionsPipeline(config)  # no GroqClient needed for this notebook

print("=" * 60)
print("✅ PIPELINE INITIALIZED")
print("=" * 60)
print(f"   Video:   {config.NOME_ORACAO}")
print(f"   Folder:  {config.pasta_oracao}")
print("=" * 60)


✅ PIPELINE INITIALIZED
   Video:   40_Matt_02
   Folder:  /content/drive/MyDrive/narrated_video/videos/40_Matt_02


In [4]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 DESCOBRIR CÓDIGOS DE LEGENDA DISPONÍVEIS NESTE VÍDEO          ║
# ║  Célula temporária — rode uma vez, veja a lista, ajuste o código  ║
# ║  do chinês em IDIOMAS_ALVO / FONTE_TEXTO_IDIOMA e prossiga.       ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess
from youtube_utils import garantir_runtime_js, resolver_cookies, garantir_yt_dlp_atualizado

garantir_yt_dlp_atualizado()
extra_args = garantir_runtime_js()
cookies = resolver_cookies(config)
extra_cookies = ["--cookies", str(cookies)] if cookies else []

resultado = subprocess.run(
    ["yt-dlp", *extra_args, "--extractor-args", "youtube:formats=missing_pot",
     *extra_cookies, "--list-subs", URL_YOUTUBE],
    capture_output=True, text=True,
)
print(resultado.stdout)
if resultado.returncode != 0:
    print("STDERR:", resultado.stderr[-1500:])

[youtube] Extracting URL: https://www.youtube.com/watch?v=4vTN7tBG3a8
[youtube] 4vTN7tBG3a8: Downloading webpage
[youtube] 4vTN7tBG3a8: Downloading android vr player API JSON
[info] Available automatic captions for 4vTN7tBG3a8:
Language Name                  Formats
ab       Abkhazian             vtt, srt, ttml, srv3, srv2, srv1, json3
aa       Afar                  vtt, srt, ttml, srv3, srv2, srv1, json3
af       Afrikaans             vtt, srt, ttml, srv3, srv2, srv1, json3
ak       Akan                  vtt, srt, ttml, srv3, srv2, srv1, json3
sq       Albanian              vtt, srt, ttml, srv3, srv2, srv1, json3
am       Amharic               vtt, srt, ttml, srv3, srv2, srv1, json3
ar       Arabic                vtt, srt, ttml, srv3, srv2, srv1, json3
hy       Armenian              vtt, srt, ttml, srv3, srv2, srv1, json3
as       Assamese              vtt, srt, ttml, srv3, srv2, srv1, json3
ay       Aymara                vtt, srt, ttml, srv3, srv2, srv1, json3
az       Azerbaijani   

In [5]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔍 LISTAR FORMATOS DE ÁUDIO — filtrar por idioma                 ║
# ║  Roda "yt-dlp -F" (com o mesmo workaround já aplicado) e filtra   ║
# ║  só as linhas de áudio, destacando as que mencionam LANG_FILTRO.  ║
# ║  Pré-requisito: já ter rodado Setup + Configuration + Initialize. ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess
from youtube_utils import garantir_runtime_js, resolver_cookies, garantir_yt_dlp_atualizado, ARGS_PO_TOKEN_WORKAROUND

LANG_FILTRO = "ko"  # troque para o código (ou nome) do idioma que quer investigar

garantir_yt_dlp_atualizado()
extra_args = garantir_runtime_js()
cookies = resolver_cookies(config)
extra_cookies = ["--cookies", str(cookies)] if cookies else []

resultado = subprocess.run(
    ["yt-dlp", *extra_args, *ARGS_PO_TOKEN_WORKAROUND, *extra_cookies, "-F", URL_YOUTUBE],
    capture_output=True, text=True,
)

print("📋 Todas as faixas de áudio disponíveis agora:\n")
linhas_audio = [l for l in resultado.stdout.splitlines() if "audio only" in l]
for linha in linhas_audio:
    print(" ", linha)

print(f"\n🔎 Linhas mencionando '{LANG_FILTRO}':\n")
encontrou = False
for linha in linhas_audio:
    if LANG_FILTRO.lower() in linha.lower():
        print("  →", linha)
        encontrou = True

if not encontrou:
    print(f"  Nenhuma faixa de áudio para '{LANG_FILTRO}' encontrada agora — pode ser que a")
    print("  dublagem automática para esse idioma simplesmente não exista (ou não")
    print("  esteja disponível) neste vídeo no momento, mesmo que a LEGENDA exista.")

if resultado.stderr.strip():
    print("\n(avisos/erros do yt-dlp, se houver)")
    for linha in resultado.stderr.splitlines():
        if "WARNING" in linha or "ERROR" in linha:
            print("  ", linha[:200])


📋 Todas as faixas de áudio disponíveis agora:

  139    m4a   audio only      2 |   1.29MiB   49k https | audio only        mp4a.40.5   49k 22k [en-US] English (US) original (default), low, m4a_dash
  249-0  webm  audio only      2 |   1.30MiB   49k https | audio only        opus        49k 48k [iw] Hebrew, low, webm_dash
  249-1  webm  audio only      2 |   1.32MiB   50k https | audio only        opus        50k 48k [ru] Russian, low, webm_dash
  249-2  webm  audio only      2 |   1.33MiB   50k https | audio only        opus        50k 48k [pl] Polish, low, webm_dash
  249-3  webm  audio only      2 |   1.33MiB   50k https | audio only        opus        50k 48k [de-DE] German (DE), low, webm_dash
  249-4  webm  audio only      2 |   1.33MiB   50k https | audio only        opus        50k 48k [id] Indonesian, low, webm_dash
  249-5  webm  audio only      2 |   1.33MiB   51k https | audio only        opus        51k 48k [ar] Arabic, low, webm_dash
  249-6  webm  audio only      2 |   1

In [6]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🔧 TESTE RÁPIDO ISOLADO — baixar 1 faixa de áudio                ║
# ║  Roda só o download (sem Whisper), pra iterar rápido até achar    ║
# ║  a combinação que funciona pra este vídeo específico.             ║
# ║  Pré-requisito: já ter rodado Setup + Configuration + Initialize  ║
# ║  do caption-multilang-sources-gather.ipynb (usa 'config' e 'URL_YOUTUBE'). ║
# ╚══════════════════════════════════════════════════════════════════╝

import subprocess
from pathlib import Path
from youtube_utils import garantir_runtime_js, resolver_cookies, garantir_yt_dlp_atualizado

LANG_TESTE = "ko"  # troque para o idioma que está falhando

garantir_yt_dlp_atualizado()
extra_args = garantir_runtime_js()
cookies = resolver_cookies(config)
extra_cookies = ["--cookies", str(cookies)] if cookies else []
print(f"Cookies: {'OK — ' + str(cookies) if cookies else 'NENHUM (pode ser parte do problema)'}")
print()

# ── Combinações a tentar, em ordem ──────────────────────────────────────────
combinacoes = {
    "1) automático puro (sem extractor-args)": [],
    "2) missing_pot": ["--extractor-args", "youtube:formats=missing_pot"],
    "3) missing_pot + tv_downgraded": ["--extractor-args", "youtube:formats=missing_pot;player_client=default,tv_downgraded"],
    "4) missing_pot + web_embedded": ["--extractor-args", "youtube:formats=missing_pot;player_client=default,web_embedded"],
    "5) missing_pot + mweb": ["--extractor-args", "youtube:formats=missing_pot;player_client=default,mweb"],
}

formato = f"ba[language^={LANG_TESTE}]/bestaudio[language^={LANG_TESTE}]"

for nome, args_extra in combinacoes.items():
    print(f"── Tentando: {nome} ──")
    cmd = [
        "yt-dlp", *extra_args, *args_extra, *extra_cookies,
        "-f", formato,
        "--extract-audio", "--audio-format", "wav",
        "-o", f"teste_{LANG_TESTE}.%(ext)s",
        URL_YOUTUBE,
    ]
    resultado = subprocess.run(cmd, capture_output=True, text=True)
    saida = Path(f"teste_{LANG_TESTE}.wav")
    if saida.exists():
        print(f"   ✅ FUNCIONOU! ({saida.stat().st_size/1_048_576:.1f} MB)")
        print(f"   Use esta combinação de extractor-args no ARGS_PO_TOKEN_WORKAROUND.")
        saida.unlink()  # limpa o arquivo de teste
        break
    else:
        ultima_linha_erro = resultado.stderr.strip().splitlines()[-1] if resultado.stderr.strip() else "(sem stderr)"
        print(f"   ❌ falhou — {ultima_linha_erro[:150]}")
    print()
else:
    print("Nenhuma combinação funcionou automaticamente.")
    print("Próximo passo: rode 'yt-dlp -F URL' e use FORMATO_MANUAL_AUDIO com o ID exato.")

Cookies: OK — cookies.txt

── Tentando: 1) automático puro (sem extractor-args) ──
   ❌ falhou — ERROR: [youtube] 4vTN7tBG3a8: Requested format is not available. Use --list-formats for a list of available formats

── Tentando: 2) missing_pot ──
   ❌ falhou — ERROR: [youtube] 4vTN7tBG3a8: Requested format is not available. Use --list-formats for a list of available formats

── Tentando: 3) missing_pot + tv_downgraded ──
   ✅ FUNCIONOU! (40.4 MB)
   Use esta combinação de extractor-args no ARGS_PO_TOKEN_WORKAROUND.


In [7]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📺 DOWNLOAD — YouTube captions (per language)                   ║
# ║  Saves {NOME}_yt_{lang}.srt to Drive. Missing languages are      ║
# ║  skipped with a warning, not a hard failure.                     ║
# ╚══════════════════════════════════════════════════════════════════╝

resultado_legendas = pipeline.baixar_legendas_youtube(URL_YOUTUBE, IDIOMAS_ALVO)
print(f"\n✅ {len(resultado_legendas)}/{len(IDIOMAS_ALVO)} languages downloaded: {list(resultado_legendas.keys())}")



✅ 4/5 languages downloaded: ['pt', 'es', 'fr', 'ko']


In [8]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  🎙️ DOWNLOAD — dubbed audio (per language)                       ║
# ║  Saves {NOME}_audio_{lang}.wav to Drive. Network-bound — fast.   ║
# ╚══════════════════════════════════════════════════════════════════╝

resultado_audio_download = pipeline.baixar_audio_idiomas(URL_YOUTUBE, IDIOMAS_ALVO)
print(f"\n✅ {len(resultado_audio_download)}/{len(IDIOMAS_ALVO)} languages downloaded: {list(resultado_audio_download.keys())}")



✅ 4/5 languages downloaded: ['pt', 'es', 'fr', 'ko']


In [9]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📝 TRANSCRIBE — Whisper over the dubbed audio (per language)    ║
# ║  Saves {NOME}_whisper_{lang}.srt to Drive. CPU-bound — can take a   ║
# ║  while (uses GPU automatically if the Colab runtime has one —    ║
# ║  Runtime → Change runtime type → Hardware accelerator → GPU).    ║
# ║  Reuses whatever audio is already local/on Drive — you can       ║
# ║  re-run just this cell without re-downloading.                    ║
# ╚══════════════════════════════════════════════════════════════════╝

resultado_transcricao = pipeline.transcrever_audio_idiomas(IDIOMAS_ALVO, modelo=MODELO_WHISPER)
print(f"\n✅ {len(resultado_transcricao)}/{len(IDIOMAS_ALVO)} languages transcribed: {list(resultado_transcricao.keys())}")


100%|███████████████████████████████████████| 139M/139M [00:01<00:00, 82.8MiB/s]



✅ 4/5 languages transcribed: ['pt', 'es', 'fr', 'ko']


In [10]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  👀 PREVIEW — what was collected, per language                   ║
# ╚══════════════════════════════════════════════════════════════════╝

from srt_utils import ler_srt

print("Sources collected per language:\n")
for lang in IDIOMAS_ALVO:
    yt_path   = Path(config.nome_srt_yt(lang))
    edge_path = Path(config.nome_srt_whisper(lang))
    yt_ok   = "✅" if lang in resultado_legendas else "❌"
    edge_ok = "✅" if lang in resultado_transcricao else "❌"
    print(f"[{lang.upper()}]  YouTube: {yt_ok} {config.nome_srt_yt(lang)}   Whisper: {edge_ok} {config.nome_srt_whisper(lang)}")

print()
for lang in IDIOMAS_ALVO:
    if lang in resultado_legendas:
        legendas = ler_srt(resultado_legendas[lang])
        print(f"\n--- {lang.upper()} (YouTube, {len(legendas)} blocks) — first 3 ---")
        for leg in legendas[:3]:
            print(f"  [{leg.inicio_str} → {leg.fim_str}]  {leg.texto}")


Sources collected per language:

[EN]  YouTube: ❌ 40_Matt_02_yt_en.srt   Whisper: ❌ 40_Matt_02_edge_en.srt
[PT]  YouTube: ✅ 40_Matt_02_yt_pt.srt   Whisper: ✅ 40_Matt_02_edge_pt.srt
[ES]  YouTube: ✅ 40_Matt_02_yt_es.srt   Whisper: ✅ 40_Matt_02_edge_es.srt
[FR]  YouTube: ✅ 40_Matt_02_yt_fr.srt   Whisper: ✅ 40_Matt_02_edge_fr.srt
[KO]  YouTube: ✅ 40_Matt_02_yt_ko.srt   Whisper: ✅ 40_Matt_02_edge_ko.srt


--- PT (YouTube, 101 blocks) — first 3 ---
  [00:00:04,520 → 00:00:09,240]  Ora, tendo Jesus nascido em Belém da
  [00:00:06,680 → 00:00:11,400]  Judeia, nos dias do rei Herodes, eis que
  [00:00:11,400 → 00:00:15,280]  vieram do oriente  a Jerusalém, dizendo:

--- ES (YouTube, 102 blocks) — first 3 ---
  [00:00:04,520 → 00:00:09,240]  Cuando Jesús nació en Belén de
  [00:00:06,680 → 00:00:11,400]  Judea, en tiempos del rey Herodes, unos
  [00:00:09,240 → 00:00:13,520]  magos de Oriente

--- FR (YouTube, 102 blocks) — first 3 ---
  [00:00:04,520 → 00:00:09,240]  Or, Jésus naquit à Bethlée

In [12]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  📥 DOWNLOAD — for manual correction                             ║
# ║  Pick which files to download below (defaults to all collected). ║
# ╚══════════════════════════════════════════════════════════════════╝

from google.colab import files

arquivos_para_baixar = list(resultado_legendas.values()) + list(resultado_transcricao.values())

if not arquivos_para_baixar:
    print("Nothing to download — no sources were collected.")
else:
    for caminho in arquivos_para_baixar:
        if caminho.suffix == ".srt":
            print(f"📥 {caminho.name}")
            files.download(str(caminho))

    print()
    print("After correcting locally, upload the files back to Drive at:")
    print(f"   {config.pasta_oracao}")
    print("(overwrite the same filenames)")


📥 40_Matt_02_yt_pt.srt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

📥 40_Matt_02_yt_es.srt


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

📥 40_Matt_02_edge_ko.srt


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


After correcting locally, upload the files back to Drive at:
   /content/drive/MyDrive/narrated_video/videos/40_Matt_02
(overwrite the same filenames)
